# MV-TransUNet

## Boundary-Aware Hybrid CNN-Vision Transformer Framework for Multi-Scale Retinal Vessel Segmentation

### Google Colab GPU Training Pipeline


This notebook provides the complete training workflow for the MV-TransUNet research framework.

## Pipeline

1. Connect Google Colab GPU
2. Clone MV-TransUNet repository
3. Install dependencies
4. Verify model architecture
5. Load retinal vessel datasets
6. Train the model
7. Save checkpoints
8. Evaluate segmentation performance


## Datasets

- DRIVE
- STARE
- CHASE_DB1
- HRF


## Model

MV-TransUNet combines:

- ResNet50 CNN feature extraction
- Vision Transformer global context modeling
- Vessel Attention Module
- Multi-scale progressive decoder
- Boundary-aware joint loss


Author:
MV-TransUNet Research Project

In [1]:
# ============================================================
# GPU VERIFICATION
# ============================================================

import torch

print("=" * 60)
print("MV-TransUNet GPU Environment Check")
print("=" * 60)


print("\nPyTorch Version:")
print(torch.__version__)


print("\nCUDA Available:")
print(torch.cuda.is_available())


if torch.cuda.is_available():

    print("\nGPU Device:")
    print(torch.cuda.get_device_name(0))


    gpu_properties = torch.cuda.get_device_properties(0)

    print("\nGPU Memory:")
    print(
        f"{gpu_properties.total_memory / 1024**3:.2f} GB"
    )


else:

    print("\nWARNING: GPU NOT DETECTED")
    print("Please reconnect Colab GPU runtime")



print("\nEnvironment check completed.")

In [2]:
# ============================================================
# CLONE MV-TRANSUNET FROM GITHUB
# ============================================================

!git clone https://github.com/JEGAH01/MV-TransUNet.git

In [30]:
import os

# Move into MV-TransUNet project
os.chdir("/content/MV-TransUNet")


print("Current directory:")
!pwd


print("\nProject structure:")
!ls

In [31]:
# ============================================================
# INSTALL MV-TRANSUNET DEPENDENCIES
# ============================================================

!pip install -q \
albumentations \
opencv-python \
scikit-image \
pyyaml \
tqdm \
tensorboard \
matplotlib \
pandas

In [5]:
# ============================================================
# VERIFY DEPENDENCIES
# ============================================================

import torch
import cv2
import albumentations
import skimage
import yaml


print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

print("\nDependencies loaded successfully")

In [7]:
# ============================================================
# MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount('/content/drive')

In [32]:
# ============================================================
# LINK DATASET
# ============================================================

import os
import shutil


SOURCE = "/content/drive/MyDrive/MV-TransUNet/datasets_processed"

TARGET = "/content/MV-TransUNet/datasets_processed"



if not os.path.exists(TARGET):

    shutil.copytree(
        SOURCE,
        TARGET
    )

    print("Dataset copied successfully")

else:

    print("Dataset already exists")



!ls datasets_processed

In [33]:
# ============================================================
# CHECK CURRENT CONFIG
# ============================================================

!cat config.yaml

In [34]:
# Print complete config without truncation

with open("config.yaml", "r") as f:
    config_text = f.read()

print(config_text)

In [35]:
import yaml

with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

print("DATASET SECTION")
print("================")
print(config["dataset"])

In [36]:
print("DATALOADER SECTION")
print("===================")
print(config["dataloader"])

In [37]:
print("OPTIMIZER SECTION")
print("==================")
print(config["optimizer"])

In [38]:
import yaml

with open("config.yaml") as f:
    config = yaml.safe_load(f)

print("Dataset:")
print(config["dataset"]["train_dataset"])

print("\nDataloader:")
print(config["dataloader"])

print("\nHardware:")
print(config["hardware"])

print("\nTraining:")
print(config["training"])

In [15]:
from google.colab import drive

drive.mount('/content/drive')

In [16]:
import os


DRIVE_DATASET = "/content/drive/MyDrive/MV-TransUNet/datasets_processed"


print("Checking dataset location...\n")


if os.path.exists(DRIVE_DATASET):

    print("Dataset found!")

    print(
        os.listdir(DRIVE_DATASET)
    )

else:

    print("Dataset not found")

In [39]:
# ============================================================
# CELL 9
# MV-TransUNet Dataset Loader Test
# ============================================================


import torch
import yaml
import os


from src.datasets import build_dataloaders



# ------------------------------------------------------------
# Load configuration
# ------------------------------------------------------------

with open("config.yaml", "r") as f:

    config = yaml.safe_load(f)



# ------------------------------------------------------------
# Dataset paths
# ------------------------------------------------------------

IMAGE_DIR = config["dataset"]["train_dataset"]["image_dir"]

MASK_DIR = config["dataset"]["train_dataset"]["mask_dir"]



print("="*60)

print("Dataset Loader Test")

print("="*60)



print("\nImage directory:")
print(IMAGE_DIR)


print("\nMask directory:")
print(MASK_DIR)



# ------------------------------------------------------------
# Check folders exist
# ------------------------------------------------------------

print("\nChecking paths...")


assert os.path.exists(IMAGE_DIR), \
    "Image directory not found"


assert os.path.exists(MASK_DIR), \
    "Mask directory not found"



print("Paths OK")



# ------------------------------------------------------------
# Build dataloader
# ------------------------------------------------------------

train_loader, val_loader = build_dataloaders(

    image_dir=IMAGE_DIR,

    mask_dir=MASK_DIR,

    image_size=config["preprocessing"]["image_size"]["height"],

    batch_size=config["dataloader"]["batch_size"],

    num_workers=config["dataloader"]["num_workers"]

)



print("\nDataLoader created")



# ------------------------------------------------------------
# Load one batch
# ------------------------------------------------------------

batch = next(iter(train_loader))



images = batch["image"]

masks = batch["mask"]



print("\nBatch information")

print("----------------------------")


print(
    "Images shape:",
    images.shape
)


print(
    "Masks shape:",
    masks.shape
)



print(
    "Images dtype:",
    images.dtype
)


print(
    "Masks dtype:",
    masks.dtype
)



# ------------------------------------------------------------
# GPU test
# ------------------------------------------------------------


device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


images = images.to(device)

masks = masks.to(device)



print("\nGPU Transfer")

print("----------------------------")


print(
    "Device:",
    images.device
)


print(
    "GPU memory allocated:",
    torch.cuda.memory_allocated()/1024**2,
    "MB"
)



print("\nDataset loader test completed successfully")

In [20]:
# ============================================================
# CELL 10
# MV-TransUNet GPU Forward Pass Test
# ============================================================


import torch

from models.mv_transunet import MVTransUNet



print("="*60)

print("MV-TransUNet GPU Forward Test")

print("="*60)



# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print("\nDevice:")

print(device)



# ------------------------------------------------------------
# Load model
# ------------------------------------------------------------

print("\nLoading model...")


model = MVTransUNet(

    pretrained=True

)



model = model.to(device)



model.eval()



print("Model loaded")



# ------------------------------------------------------------
# Get sample batch
# ------------------------------------------------------------

images = images.to(device)



print("\nInput:")

print(images.shape)



# ------------------------------------------------------------
# Forward pass
# ------------------------------------------------------------

with torch.no_grad():


    outputs = model(images)



print("\nOutput:")

print(outputs.shape)



# ------------------------------------------------------------
# Parameters
# ------------------------------------------------------------

parameters = sum(

    p.numel()

    for p in model.parameters()

    if p.requires_grad

)



print("\nTrainable parameters:")

print(parameters)



# ------------------------------------------------------------
# GPU memory
# ------------------------------------------------------------


print("\nGPU memory used:")

print(

    torch.cuda.memory_allocated()/1024**3,

    "GB"

)



print("\nForward test completed successfully")

In [21]:
from models.losses import MVTransUNetLoss

loss = MVTransUNetLoss()

print(loss)

In [22]:
!git pull

In [23]:
import os

os.chdir("/content/MV-TransUNet")

print(os.getcwd())

In [24]:
import inspect

from models.losses import BoundaryExtractor


print(
    inspect.getsource(BoundaryExtractor)
)

In [25]:
!git pull

In [26]:
import os
import sys

PROJECT_DIR = "/content/MV-TransUNet"

os.chdir(PROJECT_DIR)

sys.path.append(PROJECT_DIR)

print("Current directory:")
print(os.getcwd())

print("\nProject files:")
!ls

In [27]:
# ============================================================
# CELL 11A
# MV-TransUNet Loss Verification
# ============================================================


import torch

from models.losses import MVTransUNetLoss



print("="*60)

print("MV-TransUNet Loss Verification")

print("="*60)



# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

device = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else "cpu"

)



print("\nDevice:")

print(device)



# ------------------------------------------------------------
# Create loss function
# ------------------------------------------------------------


criterion = MVTransUNetLoss()



print("\nLoss Function:")

print(criterion)



# ------------------------------------------------------------
# Create fake prediction and target
# ------------------------------------------------------------


prediction = torch.randn(

    8,

    1,

    256,

    256,

    device=device,

    requires_grad=True

)



target = torch.randint(

    0,

    2,

    (

        8,

        1,

        256,

        256

    ),

    device=device

).float()



print("\nPrediction shape:")

print(prediction.shape)



print("\nTarget shape:")

print(target.shape)



# ------------------------------------------------------------
# Calculate loss
# ------------------------------------------------------------


loss_output = criterion(

    prediction,

    target

)



print("\nLoss output:")

print(loss_output)



# ------------------------------------------------------------
# Check total loss
# ------------------------------------------------------------


total_loss = loss_output["total_loss"]



print("\nTotal Loss:")

print(total_loss.item())



# ------------------------------------------------------------
# Backward test
# ------------------------------------------------------------


total_loss.backward()



print("\nGradient check:")


if prediction.grad is not None:

    print("Gradient computed successfully")

else:

    print("Gradient failed")



print("\nLoss verification completed successfully")

In [40]:
# ============================================================
# CELL 12A
# MV-TransUNet Optimizer + Scheduler Verification
# ============================================================


import torch
import yaml

from models.mv_transunet import MVTransUNet

from torch.optim import AdamW

from torch.optim.lr_scheduler import CosineAnnealingLR

from torch.cuda.amp import GradScaler



print("="*60)

print("MV-TransUNet Optimizer Verification")

print("="*60)



# ------------------------------------------------------------
# Load configuration
# ------------------------------------------------------------

with open("config.yaml", "r") as f:

    config = yaml.safe_load(f)



# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

device = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else "cpu"

)


print("\nDevice:")

print(device)



# ------------------------------------------------------------
# Create model
# ------------------------------------------------------------

print("\nLoading model...")


model = MVTransUNet(

    pretrained=True

).to(device)



print("Model loaded")



# ------------------------------------------------------------
# Optimizer
# ------------------------------------------------------------


optimizer = AdamW(

    model.parameters(),

    lr=config["optimizer"]["learning_rate"],

    weight_decay=config["optimizer"]["weight_decay"]

)



print("\nOptimizer:")

print(optimizer)



# ------------------------------------------------------------
# Scheduler
# ------------------------------------------------------------


scheduler = CosineAnnealingLR(

    optimizer,

    T_max=config["training"]["epochs"]

)



print("\nScheduler:")

print(scheduler)



# ------------------------------------------------------------
# Mixed Precision
# ------------------------------------------------------------


scaler = GradScaler()



print("\nAMP Scaler:")

print(scaler)



# ------------------------------------------------------------
# Check learning rate
# ------------------------------------------------------------


current_lr = optimizer.param_groups[0]["lr"]



print("\nLearning Rate:")

print(current_lr)



print("\nWeight Decay:")

print(

    optimizer.param_groups[0]["weight_decay"]

)



# ------------------------------------------------------------
# Parameter count
# ------------------------------------------------------------


trainable_params = sum(

    p.numel()

    for p in model.parameters()

    if p.requires_grad

)



print("\nTrainable Parameters:")

print(trainable_params)



print("\nOptimizer verification completed successfully")

In [42]:
# ============================================================
# Reload DataLoader for Training Test
# ============================================================


import yaml

from src.datasets import build_dataloaders



# Load config

with open("config.yaml", "r") as f:

    config = yaml.safe_load(f)



train_loader, val_loader = build_dataloaders(

    image_dir=config["dataset"]["train_dataset"]["image_dir"],

    mask_dir=config["dataset"]["train_dataset"]["mask_dir"],

    image_size=config["preprocessing"]["image_size"]["height"],

    batch_size=config["dataloader"]["batch_size"],

    num_workers=config["dataloader"]["num_workers"]

)



print("DataLoaders recreated successfully")


print("\nTraining batches:")

print(len(train_loader))


print("\nValidation batches:")

print(len(val_loader))

In [43]:
# ============================================================
# CELL 12B
# MV-TransUNet One Batch Training Dry Run
# ============================================================


import torch

from torch.amp import autocast, GradScaler

from models.mv_transunet import MVTransUNet

from models.losses import MVTransUNetLoss

from torch.optim import AdamW



print("="*60)

print("MV-TransUNet One Batch Training Test")

print("="*60)



device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)



# ------------------------------------------------------------
# Load one batch
# ------------------------------------------------------------

batch = next(iter(train_loader))


images = batch["image"].to(
    device,
    non_blocking=True
)


masks = batch["mask"].to(
    device,
    non_blocking=True
)



print("\nInput:")
print(images.shape)


print("\nMask:")
print(masks.shape)



# ------------------------------------------------------------
# Model
# ------------------------------------------------------------

model = MVTransUNet(
    pretrained=True
).to(device)


model.train()



# ------------------------------------------------------------
# Loss
# ------------------------------------------------------------

criterion = MVTransUNetLoss()



# ------------------------------------------------------------
# Optimizer
# ------------------------------------------------------------

optimizer = AdamW(

    model.parameters(),

    lr=0.0001,

    weight_decay=0.0001

)



scaler = GradScaler("cuda")



# ------------------------------------------------------------
# Forward + Backward
# ------------------------------------------------------------


optimizer.zero_grad()



with autocast("cuda"):


    outputs = model(images)


    loss_dict = criterion(

        outputs,

        masks

    )


    loss = loss_dict["total_loss"]



print("\nOutput:")
print(outputs.shape)



print("\nLoss:")
print(loss.item())



# Backward

scaler.scale(loss).backward()



scaler.step(

    optimizer

)


scaler.update()



print("\nOptimizer step completed")


print("\nGPU memory:")

print(

    torch.cuda.memory_allocated()/1024**3,

    "GB"

)



print("\nOne batch training test completed successfully")

In [5]:
!git pull


In [44]:
!python -m src.train

In [33]:
from src.datasets import build_dataloaders
import yaml


with open("config.yaml") as f:
    config = yaml.safe_load(f)


train_loader, val_loader = build_dataloaders(
    image_dir=config["dataset"]["train_dataset"]["image_dir"],
    mask_dir=config["dataset"]["train_dataset"]["mask_dir"],
    image_size=256,
    batch_size=8,
    num_workers=2
)


print("="*60)
print("Dataset Split Verification")
print("="*60)

print(
    "Training images:",
    len(train_loader.dataset)
)

print(
    "Validation images:",
    len(val_loader.dataset)
)

print(
    "Training batches:",
    len(train_loader)
)

print(
    "Validation batches:",
    len(val_loader)
)

In [34]:
# ============================================================
# CHECK ACTUAL DRIVE FILES USED BY THE DATASET
# ============================================================

from pathlib import Path
import yaml

from src.datasets import RetinalVesselDataset, get_validation_transform


with open("config.yaml", "r") as file:
    config = yaml.safe_load(file)


image_dir = Path(
    config["dataset"]["train_dataset"]["image_dir"]
)

mask_dir = Path(
    config["dataset"]["train_dataset"]["mask_dir"]
)


print("=" * 60)
print("DRIVE DATASET FILE CHECK")
print("=" * 60)

print("\nImage directory:")
print(image_dir.resolve())

print("\nMask directory:")
print(mask_dir.resolve())


image_files = sorted(
    file
    for file in image_dir.iterdir()
    if file.is_file()
)

mask_files = sorted(
    file
    for file in mask_dir.iterdir()
    if file.is_file()
)


print("\nFiles physically present")
print("------------------------")
print("Images:", len(image_files))
print("Masks :", len(mask_files))


dataset = RetinalVesselDataset(
    image_dir=str(image_dir),
    mask_dir=str(mask_dir),
    transform=get_validation_transform(256),
    clahe=True
)


print("\nDataset object")
print("------------------------")
print("Loaded pairs:", len(dataset))


print("\nFirst five images:")
for file in image_files[:5]:
    print(file.name)


print("\nFirst five masks:")
for file in mask_files[:5]:
    print(file.name)

In [37]:
!git pull

In [39]:
import os

print(os.getcwd())

In [40]:
!grep -n "validation_ratio" config.yaml

In [46]:
!git pull

In [43]:
import os
print(os.getcwd())

In [44]:
!grep -n "validation_ratio" config.yaml

In [19]:
!git status

In [47]:
import yaml

with open("config.yaml", "r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

print("Config loaded successfully")
print("Batch size:", config["dataloader"]["batch_size"])
print("Validation ratio:", config["dataset"]["validation_ratio"])
print("Accumulation:", config["training"]["gradient_accumulation_steps"])
print("Resume:", config["checkpoint"]["resume"])
print("Checkpoint:", config["checkpoint"]["directory"])

In [3]:
from src.datasets import build_dataloaders

train_loader, val_loader = build_dataloaders(
    image_dir=config["dataset"]["train_dataset"]["image_dir"],
    mask_dir=config["dataset"]["train_dataset"]["mask_dir"],
    image_size=config["preprocessing"]["image_size"]["height"],
    batch_size=config["dataloader"]["batch_size"],
    validation_ratio=config["dataset"]["validation_ratio"],
    num_workers=config["dataloader"]["num_workers"],
    seed=config["dataset"]["split_seed"],
)

print("=" * 60)
print("DATASET SPLIT")
print("=" * 60)

print("Training images :", len(train_loader.dataset))
print("Validation images:", len(val_loader.dataset))
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

In [20]:
from src.datasets import build_dataloaders
import inspect

print(inspect.signature(build_dataloaders))

In [17]:
from src.datasets import build_dataloaders

train_loader, val_loader = build_dataloaders(
    image_dir=config["dataset"]["train_dataset"]["image_dir"],
    mask_dir=config["dataset"]["train_dataset"]["mask_dir"],
    image_size=config["preprocessing"]["image_size"]["height"],
    batch_size=config["dataloader"]["batch_size"],
    validation_ratio=config["dataset"]["validation_ratio"],
    num_workers=config["dataloader"]["num_workers"],
    clahe=config["preprocessing"]["clahe"]["enabled"],
)

print("=" * 60)
print("DATASET SPLIT")
print("=" * 60)

print("Training images :", len(train_loader.dataset))
print("Validation images:", len(val_loader.dataset))
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

In [19]:
!git pull

In [1]:
import inspect
from src.datasets import build_dataloaders

print(inspect.signature(build_dataloaders))

In [21]:
train_loader, val_loader = build_dataloaders(
    image_dir=config["dataset"]["train_dataset"]["image_dir"],
    mask_dir=config["dataset"]["train_dataset"]["mask_dir"],
    image_size=config["preprocessing"]["image_size"]["height"],
    batch_size=config["dataloader"]["batch_size"],
    validation_ratio=config["dataset"]["validation_ratio"],
    num_workers=config["dataloader"]["num_workers"],
    clahe=config["preprocessing"]["clahe"]["enabled"],
    seed=config["dataset"]["split_seed"],
)

print("Training images :", len(train_loader.dataset))
print("Validation images:", len(val_loader.dataset))
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

In [23]:
!git pull

In [22]:
!python -m src.train

In [ ]:
import yaml

with open("config.yaml", "r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

print("=" * 60)
print("A100 EXPERIMENT CONFIGURATION")
print("=" * 60)

print("GPU:", config["hardware"]["gpu_type"])
print(
    "Image size:",
    config["preprocessing"]["image_size"]
)
print(
    "Transformer image size:",
    config["model"]["transformer"]["image_size"]
)
print(
    "Batch size:",
    config["dataloader"]["batch_size"]
)
print(
    "Gradient accumulation:",
    config["training"]["gradient_accumulation_steps"]
)
print(
    "Effective batch size:",
    config["dataloader"]["batch_size"]
    * config["training"]["gradient_accumulation_steps"]
)
print(
    "Auxiliary weights:",
    config["loss"]["auxiliary_weights"]
)
print(
    "Resume:",
    config["checkpoint"]["resume"]
)
print(
    "Checkpoint directory:",
    config["checkpoint"]["directory"]
)